In [1]:
import pandas as pd

In [2]:
df = pd.read_excel('lisabeauty.xlsx')

In [11]:
df.head(10)

,Product ID,Product Name,Product Type,Category,Brand Name,Product Line Name,Ingredients,Use Instructions,Package Size,Product Description,...,Key Ingriedients,Key Ingriedients2,Key Ingriedients3,Key Ingriedients4,Key Ingriedients5,Product Images,Verification Status,Verification Date,"Notes (For internal use: flags, manual checks, comments, etc.)",Source
0,1,Skin Light Body Lotion With Carrot Extract and...,skin care,Lotion,Skin Light,NaN,NaN,NaN,16.9 oz,Skin Light Body Lotion With Carrot Extract and...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/skin-...
1,2,Aveeno Positively Radiant Skin Brightening Exf...,skin care,Scrub,Aveeno Positively,NaN,NaN,NaN,7 oz,Aveeno Positively Radiant Skin Brightening Exf...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/aveen...
2,3,Pr. Francoise Bedon Supreme Lightening Beauty ...,skin care,Cream,Pr. Francoise,NaN,NaN,NaN,1.69 oz,Pr. Francoise Bedon Supreme Lightening Beauty ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/pr-fr...
3,4,Nutriclair Carrot Clarifying Moisturizing Milk...,skin care,Milk,Nutriclair Carrot,NaN,NaN,NaN,17 oz,Nutriclair Carrot Clarifying Moisturizing Milk...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/nutri...
4,5,Easy White Express Carrot Radiance and Clarity...,skin care,Lotion,Easy White,NaN,NaN,NaN,16.9 oz,Easy White Express Carrot Radiance Body Lotion...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/easy-...
5,6,Clear Quick Active Gel 3.45 oz,skin care,Gel,Clear Quick,NaN,NaN,NaN,3.45 oz,Clear Quick Active Gel 3.45 oz This important ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/clear...
6,7,Pure Skin Black Spot Corrector Lotion 1 oz,skin care,Lotion,Pure Skin,NaN,NaN,NaN,1 oz,Pure Skin Black Spot Corrector Lotion 1 oz Dar...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/pure-...
7,8,Clean & Clear Morning Burst Skin Brightening F...,skin care,Oil,Clean &,NaN,NaN,NaN,5 oz,Clean & Clear Morning Burst Skin Brightening F...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/clean...
8,9,G & G Dynamiclair Beauty Soap 6.7 oz / 200g,skin care,Soap,G &,NaN,NaN,NaN,"6.7 oz, 200g",G & G Dynamiclair Beauty Soap 6.7 oz This is a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/g-g-d...
9,10,Pr. Francoise Bedon Homme Lightening Lotion fo...,skin care,Lotion,Pr. Francoise,NaN,NaN,NaN,16.8 oz,Pr. Francoise Bedon Homme Lightening Lotion fo...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,https://www.lisabeautysupply.com/product/pr-fr...


In [6]:
def extract_brand(product_name: str) -> str:
    if not isinstance(product_name, str) or not product_name.strip():
        return ""
    words = product_name.strip().split()
    return " ".join(words[:2])

In [7]:
df["Brand Name"] = df["Product Name"].apply(extract_brand)

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 30 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   Product ID                                                      80 non-null     int64  
 1   Product Name                                                    80 non-null     object 
 2   Product Type                                                    80 non-null     object 
 3   Category                                                        68 non-null     object 
 4   Brand Name                                                      80 non-null     object 
 5   Product Line Name                                               0 non-null      float64
 6   Ingredients                                                     0 non-null      float64
 7   Use Instructions                                       

In [4]:
import pandas as pd

# Flat list of skincare forms
skincare_forms = [
    "Cream", "Lotion", "Gel", "Foam", "Oil", "Serum", "Balm", 
    "Sheet Mask", "Clay Mask", "Overnight Mask", "Toner", "Scrub", 
    "Exfoliant", "Eye Cream", "Lip Balm", "Micellar Water", "Sunscreen", "Milk","Ointment","Soap", "Essence", "Exfoliant"
]

# Function to find the first matching skincare form
def get_skincare_form(product_name):
    if pd.isna(product_name):
        return None
    product_name_lower = product_name.lower()
    for form in skincare_forms:
        form_lower = form.lower()
        if product_name_lower.startswith(form_lower) or form_lower in product_name_lower:
            return form  # return the matched form
    return None

# Ensure 'Category' exists
if 'Category' not in df.columns:
    df['Category'] = None

# Update 'Category' with the actual skincare form
df['Category'] = df['Product Name'].apply(get_skincare_form)


In [9]:
from datetime import datetime
df['Date Added'] = datetime.today().date()

In [10]:
df['Product ID'] = range(1, len(df) + 1)

In [12]:
df.to_csv('lisa.csv', index = False)

In [13]:
import os
import re
import requests
import pandas as pd
from time import sleep
from random import uniform
from urllib.parse import urlparse

# --- File paths ---
INPUT_FILE = "lisa.csv"             # Your CSV input file
CHECKPOINT_FILE = "products_with_images.csv"  # Backup file for resuming
IMAGES_FOLDER = "images"

# --- Setup ---
os.makedirs(IMAGES_FOLDER, exist_ok=True)

def get_headers(url):
    """Generate headers with User-Agent and Referer from the image URL."""
    parsed = urlparse(url)
    domain = f"{parsed.scheme}://{parsed.hostname}" if parsed.scheme else ""
    return {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/139.0.0.0 Safari/537.36"
        ),
        "Referer": domain,
    }

def clean_url(url):
    """Force highest quality by stripping query strings & known resize params."""
    if not url:
        return url
    url = url.split("?")[0]  # remove ?v=123&width=500
    url = re.sub(r"(_(small|compact|medium|large|grande|x\d+))", "_master", url)
    return url.strip()

def safe_filename(name):
    """Sanitize product name for filenames (no special chars)."""
    name = re.sub(r"[^\w\s-]", "", str(name))
    name = re.sub(r"\s+", "_", name)
    return name.strip("_").lower()

def download_image(url, filename):
    """Download a single image and return saved filename."""
    try:
        url = clean_url(url)
        ext = url.split(".")[-1].lower()
        if len(ext) > 5 or "/" in ext:
            ext = "jpg"
        filepath = os.path.join(IMAGES_FOLDER, f"{filename}.{ext}")

        if os.path.exists(filepath):  # already exists
            return os.path.basename(filepath)

        response = requests.get(url, headers=get_headers(url), timeout=30)
        response.raise_for_status()
        with open(filepath, "wb") as f:
            f.write(response.content)

        return os.path.basename(filepath)
    except Exception as e:
        print(f"❌ Failed to download {url}: {e}")
        return None

def process_row_images(url_list, product_name, existing_files=None):
    """
    Download valid images for a row and return filenames as comma-separated string.
    Filters:
      - Exclude URLs containing 'logo', 'icon', 'placeholder'
    """
    if pd.isna(url_list):
        return None

    cleaned = str(url_list).strip().strip("[]").replace("'", "").replace('"', "")
    urls = [u.strip() for u in re.split(r",\s*", cleaned) if u.strip()]

    base_name = safe_filename(product_name)
    saved_files = []

    # Filter unwanted words
    urls = [u for u in urls if not any(bad in u.lower() for bad in ["logo", "icon", "placeholder"])]

    for i, url in enumerate(urls, start=1):
        filename = f"{base_name}_{i}" if len(urls) > 1 else base_name
        if existing_files:
            expected_file = f"{filename}.jpg"
            if expected_file in existing_files and os.path.exists(os.path.join(IMAGES_FOLDER, expected_file)):
                saved_files.append(expected_file)
                continue

        saved = download_image(url, filename)
        if saved:
            saved_files.append(saved)
        sleep(uniform(0.2, 0.6))

    return ", ".join(saved_files) if saved_files else None

# --- Load dataset ---
if os.path.exists(CHECKPOINT_FILE):
    print(f"📂 Resuming from {CHECKPOINT_FILE}")
    df = pd.read_csv(CHECKPOINT_FILE)
else:
    print(f"📥 Loading fresh data from {INPUT_FILE}")
    df = pd.read_csv(INPUT_FILE)

if "Product Image" not in df.columns:
    df["Product Image"] = None

SAVE_EVERY = 20

for idx, row in df.iterrows():
    existing_files = None
    if pd.notna(row["Product Image"]):
        existing_files = [f.strip() for f in str(row["Product Image"]).split(",") if f.strip()]
        missing = [f for f in existing_files if not os.path.exists(os.path.join(IMAGES_FOLDER, f))]
        if not missing:
            continue
        else:
            print(f"⚠️ Missing files for row {idx}, re-downloading...")

    df.at[idx, "Product Image"] = process_row_images(
        row.get("Product Image URL"), row.get("Product Name", f"product_{idx}"),
        existing_files
    )

    if idx % SAVE_EVERY == 0:
        df.to_csv(CHECKPOINT_FILE, index=False)
        print(f"💾 Checkpoint saved at row {idx}")

df.to_csv(CHECKPOINT_FILE, index=False)
print(f"🎉 Finished! All images processed. Backup saved to {CHECKPOINT_FILE}")

📥 Loading fresh data from lisa.csv
💾 Checkpoint saved at row 0
❌ Failed to download https://www.lisabeautysupply.com/wp-content/uploads/https://ae01.alicdn.com/kf/H18b3454152624f3b816bd331061fcbf1n/Electric-Neck-Massager-Pulse-Back-6-Modes-Power-Control-Far-Infrared-Heating-Pain-Relief-Tool-Health.jpg_640x640.jpg: HTTPSConnectionPool(host='www.lisabeautysupply.com', port=443): Read timed out. (read timeout=30)
❌ Failed to download https://www.lisabeautysupply.com/wp-content/uploads/https://ae01.alicdn.com/kf/H8f85b80101f64292b7d93c3f8262b82e3/Electric-Neck-Massager-Pulse-Back-6-Modes-Power-Control-Far-Infrared-Heating-Pain-Relief-Tool-Health.jpg: HTTPSConnectionPool(host='www.lisabeautysupply.com', port=443): Read timed out. (read timeout=30)
❌ Failed to download https://www.lisabeautysupply.com/wp-content/uploads/https://ae01.alicdn.com/kf/H620e8e5ab0014e3a8d8d72713250fb69A/Electric-Neck-Massager-Pulse-Back-6-Modes-Power-Control-Far-Infrared-Heating-Pain-Relief-Tool-Health.jpg_640x640

In [40]:
df.columns

Index(['Product ID', 'Product Name', 'Product Type', 'Category', 'Brand Name',
       'Product Line Name', 'Ingredients', 'Use Instructions', 'Package Size',
       'Product Description', 'Product Colour', 'Country of Origin',
       'Date Added', 'Barcode (EAN/UPC)', 'Barcode Type (e.g., EAN-13, UPC-A)',
       'Batch Number', 'SKU', 'Benefits', 'Product Image URL',
       'Hero Ingriedients Match', 'Key Ingriedients', 'Key Ingriedients2',
       'Key Ingriedients3', 'Key Ingriedients4', 'Key Ingriedients5',
       'Product Images', 'Verification Status', 'Verification Date',
       'Notes (For internal use: flags, manual checks, comments, etc.)',
       'Source'],
      dtype='object')

In [43]:
duplicate_product_names = df[df.duplicated(subset=["Product Name"], keep=False)]
print(f"Number of duplicate product names: {len(duplicate_product_names)}")
print(len(duplicate_product_names))

Number of duplicate product names: 0
0


In [44]:
df.to_excel("lisa_beauty.xlsx", index=False)